# Guess the Output 🎯

Each question describes an experiment. **Commit to a guess first** (write it in `my_guess`), *then* run the experiment cell.
Wrong guesses are the point: the surprise is what makes it stick.

Keep score at the bottom. 10/12 or more = intuition of a seasoned practitioner.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, warnings
warnings.filterwarnings("ignore")
rng = np.random.default_rng(42)
score = {"right": 0, "asked": 0}
def reveal(guess, answer, why):
    score["asked"] += 1
    if guess is None:
        print(f"(no guess entered) The answer is {answer}.")
    elif str(guess).strip().upper() == answer:
        score["right"] += 1; print(f"✅ Correct: {answer}!")
    else:
        print(f"❌ You said {guess}, the answer is {answer}.")
    print("Why:", why)

---
## Q1. Learning rate × 10
Gradient descent fits a linear regression on one standardised feature. With learning rate 0.15 the loss falls smoothly.

🤔 **What does the loss curve look like with learning rate 1.5?**

- **A** Same shape, just reaches the minimum 10× faster
- **B** Slower, because big steps overshoot a little
- **C** It grows instead of falling – training diverges
- **D** It converges to a lower loss than before

Related: [[Gradient Descent]]

In [ ]:
my_guess = None   # ← write "A", "B", "C" or "D" BEFORE running the next cell

In [ ]:
x = rng.normal(size=200); x = (x - x.mean()) / x.std(); y = 3 * x + rng.normal(0, 0.5, 200)
def run(lr, steps=30):
    w, L = 0.0, []
    for _ in range(steps):
        L.append(np.mean((w * x - y) ** 2)); w -= lr * np.mean(2 * (w * x - y) * x)
    return L
for lr in [0.15, 1.5]: plt.semilogy(run(lr), label=f"lr={lr}")
plt.xlabel("step"); plt.ylabel("MSE (log scale)"); plt.legend(); plt.show()
reveal(my_guess, "C", """For MSE the curvature here is 2 (twice the feature variance). Gradient descent is only stable for lr < 2/curvature = 1. At 1.5 every step overshoots further than the last, so the loss grows exponentially.""")

---
## Q2. Batch size 1
Logistic regression trained with full-batch gradient descent gives a smooth loss curve.

🤔 **What happens to the training-loss curve with mini-batches of size 1 (pure SGD), same number of updates?**

- **A** Identical curve
- **B** Noisy and jagged, but trending down
- **C** Smooth but higher
- **D** It never decreases

Related: [[Gradient Descent]]

In [ ]:
my_guess = None   # ← write "A", "B", "C" or "D" BEFORE running the next cell

In [ ]:
from sklearn.datasets import make_classification
X, y = make_classification(1000, 5, random_state=0)
def train(batch, steps=300, lr=0.1):
    w, L = np.zeros(5), []
    for _ in range(steps):
        L.append(np.mean(np.log1p(np.exp(-(2*y-1) * (X @ w)))))
        i = rng.choice(len(X), batch, replace=False); p = 1 / (1 + np.exp(-X[i] @ w))
        w -= lr * X[i].T @ (p - y[i]) / batch
    return L
plt.plot(train(1000), label="full batch"); plt.plot(train(1), label="batch = 1", alpha=.7)
plt.xlabel("update"); plt.ylabel("training loss"); plt.legend(); plt.show()
reveal(my_guess, "B", """Each single-example gradient is a noisy estimate of the true gradient, so individual steps sometimes go the wrong way. On average they point downhill, so the curve still trends down. That noise is also why SGD can escape sharp minima.""")

---
## Q3. 1-nearest-neighbour training accuracy
A 1-NN classifier is fit on the breast-cancer dataset (569 patients).

🤔 **What is its accuracy on the training set itself?**

- **A** About 50 %
- **B** About 90 %
- **C** 100 %
- **D** It depends heavily on the random seed

Related: [[k-Nearest Neighbors]]

In [ ]:
my_guess = None   # ← write "A", "B", "C" or "D" BEFORE running the next cell

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
Xb, yb = load_breast_cancer(return_X_y=True)
knn = KNeighborsClassifier(1).fit(Xb, yb)
print("train accuracy:", knn.score(Xb, yb)); print("cross-validated:", cross_val_score(KNeighborsClassifier(1), Xb, yb, cv=5).mean().round(3))
reveal(my_guess, "C", """Every training point's nearest neighbour is itself (distance 0), so it always predicts its own label. Training accuracy says nothing about generalisation – the cross-validated score is the honest number.""")

---
## Q4. Huge regularisation
Ridge regression on the diabetes dataset, with alpha raised from 1 to 1,000,000.

🤔 **What do the predictions look like?**

- **A** Almost identical to alpha = 1
- **B** Wildly oscillating
- **C** Nearly constant – everyone gets about the mean of y
- **D** All exactly zero

Related: [[Overfitting and Regularization]]

In [ ]:
my_guess = None   # ← write "A", "B", "C" or "D" BEFORE running the next cell

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Ridge
Xd, yd = load_diabetes(return_X_y=True)
for a in [1, 1e6]:
    m = Ridge(alpha=a).fit(Xd, yd); p = m.predict(Xd)
    print(f"alpha={a:>9}: |coef| sum={np.abs(m.coef_).sum():8.2f}, prediction std={p.std():6.2f}, mean={p.mean():.1f}  (mean of y = {yd.mean():.1f})")
reveal(my_guess, "C", """The penalty shrinks every coefficient towards 0, but the intercept is not penalised. With huge alpha only the intercept remains, and it equals the mean of y. Not D: the intercept keeps predictions at the mean, not 0.""")

---
## Q5. Adding noise features
A linear regression with 5 real features on 60 samples. We add 40 columns of pure random noise.

🤔 **What happens to R² on the training set and on the test set?**

- **A** Both go down
- **B** Both go up
- **C** Train goes up, test goes down
- **D** Neither changes – the model ignores noise

Related: [[Bias-Variance Tradeoff]]

In [ ]:
my_guess = None   # ← write "A", "B", "C" or "D" BEFORE running the next cell

In [ ]:
from sklearn.linear_model import LinearRegression
def data(n):
    X = rng.normal(size=(n, 5)); return X, X @ np.array([1, 2, -1, .5, 3]) + rng.normal(0, 2, n)
Xtr, ytr = data(60); Xte, yte = data(1000)
noise = lambda n: rng.normal(size=(n, 40))
m1 = LinearRegression().fit(Xtr, ytr)
Ntr, Nte = noise(60), noise(1000)
m2 = LinearRegression().fit(np.c_[Xtr, Ntr], ytr)
print(f"5 features : train R² {m1.score(Xtr, ytr):.3f}, test R² {m1.score(Xte, yte):.3f}")
print(f"+40 noise  : train R² {m2.score(np.c_[Xtr, Ntr], ytr):.3f}, test R² {m2.score(np.c_[Xte, Nte], yte):.3f}")
reveal(my_guess, "C", """Least squares can only use extra columns to fit the training data better, so training R² never decreases. Here it fits noise, which hurts on new data. That is why adjusted R², validation sets and regularisation exist.""")

---
## Q6. Copy the data 10 times
Ordinary least squares on a dataset, then on the same dataset with every row duplicated 10 times.

🤔 **What happens to the fitted coefficients?**

- **A** They become 10× larger
- **B** They stay exactly the same
- **C** They become more accurate
- **D** They become 10× smaller

Related: [[Linear Regression]]

In [ ]:
my_guess = None   # ← write "A", "B", "C" or "D" BEFORE running the next cell

In [ ]:
X = rng.normal(size=(50, 3)); y = X @ np.array([1., -2., .5]) + rng.normal(0, 1, 50)
c1 = LinearRegression().fit(X, y).coef_; c10 = LinearRegression().fit(np.tile(X, (10, 1)), np.tile(y, 10)).coef_
print("original  :", c1.round(4)); print("10× copies:", c10.round(4))
reveal(my_guess, "B", """Duplicating every row multiplies XᵀX and Xᵀy by 10, and the 10s cancel in (XᵀX)⁻¹Xᵀy. No new information was added. Note: the reported standard errors would shrink by √10 – falsely, because the copies are not independent. With regularisation the copies *would* change the fit, since the penalty no longer scales with the data.""")

---
## Q7. PCA on unscaled data
Two features: height in metres (std ≈ 0.1) and weight in grams (std ≈ 15,000).

🤔 **Without scaling, which direction is the first principal component?**

- **A** Mostly height
- **B** Mostly weight
- **C** Exactly 45° between them
- **D** Random each run

Related: [[PCA]]

In [ ]:
my_guess = None   # ← write "A", "B", "C" or "D" BEFORE running the next cell

In [ ]:
from sklearn.decomposition import PCA
h = rng.normal(1.75, .1, 500); w = 70000 + 120000 * (h - 1.75) + rng.normal(0, 12000, 500)
pca = PCA().fit(np.c_[h, w])
print("first component (height, weight):", pca.components_[0].round(6)); print("explained variance ratio:", pca.explained_variance_ratio_.round(6))
reveal(my_guess, "B", """PCA maximises variance, and variance depends on units: in grams, weight has a variance about 10¹⁰ times larger. The first component is almost pure weight. Standardise first if the units are arbitrary.""")

---
## Q8. k-means with k = n
k-means on 30 distinct points with k = 30.

🤔 **What is the final SSE (sum of squared distances to the nearest centroid)?**

- **A** Exactly 0, at least with k-means++ initialisation
- **B** About half the total variance
- **C** Larger than with k = 3
- **D** Undefined

Related: [[Clustering]]

In [ ]:
my_guess = None   # ← write "A", "B", "C" or "D" BEFORE running the next cell

In [ ]:
from sklearn.cluster import KMeans
P = rng.normal(size=(30, 2))
for k in [3, 30]: print(f"k={k:>2}: SSE = {KMeans(k, n_init=10, random_state=0).fit(P).inertia_:.6f}")
reveal(my_guess, "A", """With as many centroids as points, each point can be its own centroid at distance 0. SSE always decreases as k grows, so you cannot choose k by minimising SSE. Use the elbow, silhouette or BIC instead.""")

---
## Q9. Softmax temperature → 0
Scores z = [2.0, 1.0, 0.5]. We compute softmax(z / T).

🤔 **What happens to the probabilities as T → 0?**

- **A** They become uniform (1/3 each)
- **B** They approach one-hot: [1, 0, 0]
- **C** They don't change – softmax is scale invariant
- **D** They become negative

Related: [[Attention Mechanism]]

In [ ]:
my_guess = None   # ← write "A", "B", "C" or "D" BEFORE running the next cell

In [ ]:
z = np.array([2.0, 1.0, 0.5])
for T in [10, 1, 0.3, 0.05]:
    e = np.exp(z / T - (z / T).max()); print(f"T={T:>5}: {np.round(e / e.sum(), 3)}")
reveal(my_guess, "B", """Dividing by a small T stretches the gaps between scores, so the largest one dominates exponentially. A large T squashes the gaps towards uniform. Softmax is invariant to *adding* a constant, not to *multiplying*. LLM sampling temperature and attention's 1/√d scaling use exactly this effect.""")

---
## Q10. All-zero initialisation
A 2→4→1 network with sigmoid hidden units. Every weight and bias starts at exactly 0, then trains with gradient descent.

🤔 **What do the 4 hidden units look like after training?**

- **A** Four different useful feature detectors
- **B** All four are identical copies of each other
- **C** Training crashes with NaN
- **D** Only one unit learns, the others stay at 0

Related: [[Training Tricks]]

In [ ]:
my_guess = None   # ← write "A", "B", "C" or "D" BEFORE running the next cell

In [ ]:
X = rng.normal(size=(200, 2)); y = (X[:, 0] * X[:, 1] > 0).astype(float)
W1, b1, W2, b2 = np.zeros((2, 4)), np.zeros(4), np.zeros(4), 0.0
sig = lambda t: 1 / (1 + np.exp(-t))
for _ in range(2000):
    H = sig(X @ W1 + b1); p = sig(H @ W2 + b2); d = (p - y) / len(y)
    dH = np.outer(d, W2) * H * (1 - H)
    W2 -= 1.0 * H.T @ d; b2 -= 1.0 * d.sum(); W1 -= 1.0 * X.T @ dH; b1 -= 1.0 * dH.sum(0)
print("hidden-unit weight vectors (columns):"); print(W1.round(4)); print("output weights:", W2.round(4))
print("accuracy:", ((p > .5) == y).mean())
reveal(my_guess, "B", """Identical units receive identical gradients, so they stay identical forever (symmetry is never broken). The network effectively has one hidden unit and cannot solve this XOR-like task. Random initialisation breaks the symmetry.""")

---
## Q11. Deep tree: train vs test
Decision trees on the breast-cancer data with max_depth from 1 to 15.

🤔 **What happens to train and test accuracy as depth grows?**

- **A** Both rise steadily together
- **B** Train climbs to 100 %, test levels off below it
- **C** Test rises above train
- **D** Both fall

Related: [[Decision Trees]]

In [ ]:
my_guess = None   # ← write "A", "B", "C" or "D" BEFORE running the next cell

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
a, b, ya, yb_ = train_test_split(Xb, yb, test_size=0.3, random_state=1)
depths = range(1, 16); tr, te = [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(a, ya); tr.append(t.score(a, ya)); te.append(t.score(b, yb_))
plt.plot(depths, tr, "o-", label="train"); plt.plot(depths, te, "o-", label="test"); plt.xlabel("max_depth"); plt.legend(); plt.show()
reveal(my_guess, "B", """A deep enough tree can isolate every training point (100 % train accuracy), but past a moderate depth the extra splits fit noise. The growing gap between the curves is overfitting. Pruning, a depth limit or ensembles (random forests) fix it.""")

---
## Q12. Correlated duplicate feature
Linear regression with feature x. We add a second column x + tiny noise (correlation 0.9999).

🤔 **What happens to the two coefficients across different random samples?**

- **A** Each is about half the original coefficient, every time
- **B** They become huge and opposite-signed, changing wildly between samples – but predictions stay fine
- **C** Predictions break completely
- **D** The second coefficient is always exactly 0

Related: [[Linear Regression]]

In [ ]:
my_guess = None   # ← write "A", "B", "C" or "D" BEFORE running the next cell

In [ ]:
for seed in range(4):
    r = np.random.default_rng(seed); x = r.normal(size=100); y = 2 * x + r.normal(0, .5, 100)
    X2 = np.c_[x, x + r.normal(0, 0.01, 100)]; m = LinearRegression().fit(X2, y)
    print(f"sample {seed}: coefs = {m.coef_.round(2)}, sum = {m.coef_.sum():.2f}, train R² = {m.score(X2, y):.3f}")
reveal(my_guess, "B", """With near-duplicate columns only the *sum* of the coefficients is well determined; the split between them is driven by noise (multicollinearity, ill-conditioned XᵀX). Predictions are fine, but individual coefficients are not interpretable. Ridge regression stabilises them.""")

In [ ]:
print(f"Final score: {score['right']}/{score['asked']}")

Want more? Write your own question about something you're unsure of – predicting first, then testing, is how researchers build intuition.